### Content Splitting

In [1]:
import re
import numpy as np
from typing import Dict, Any, List, Optional

# LlamaIndex Core Imports (v0.10+)
from llama_index.core.llama_pack import BaseLlamaPack
from llama_index.core.schema import Document
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.bridge.pydantic import Field
from llama_index.core.node_parser.interface import MetadataAwareTextSplitter

# Specific Integrations
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.embeddings import BaseEmbedding
from llama_index.llms.groq import Groq

import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# Replaces OpenAI settings
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = Groq(model="llama-3.3-70b-versatile", api_key="GROQ_API_KEY")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# ⚖️ Graph-Augmented Legal RAG: Project Notes & Architecture

### 🛠 Tech Stack & Decisions
* **Framework:** `LlamaIndex` (v0.10+). Chosen for its "Data-First" architecture, offering native support for Knowledge Graphs and Semantic Chunking that is more specialized for retrieval than LangChain.
* **LLM:** `Groq` (specifically `llama-3.3-70b-versatile`). Replaces OpenAI to provide high-speed inference (LPUs) and high-quality German legal reasoning.
* **Embeddings:** `Hugging Face` (Local). Replaces OpenAI embeddings to allow for local, cost-free processing using models like `BAAI/bge-small-en-v1.5`.

---

### 🧩 Core Logic: Semantic Chunking (`base.py`)
Unlike standard recursive character splitting, this pipeline splits text based on **meaning shifts**:
1.  **Sentence Splitting:** Uses Regex to break text into individual sentences.
2.  **Context Buffering:** Surrounds each sentence with neighboring sentences to provide the embedding model with more context for better semantic representation.
3.  **Semantic Distance:** Calculates the **Cosine Distance** between the embeddings of consecutive sentence groups.
4.  **Breakpoint Identification:** Uses **NumPy Percentiles** (default 95th) to find "outlier" distances. These outliers represent significant shifts in topic where a new chunk should start.

---

### 🤖 Multi-Agent Prompting Strategy
The system uses a specialized chain of simulated experts to ensure legal accuracy:
* **Data Servant:** Performs the retrieval and lists raw findings.
* **Junior Law Consultant:** Formulates the first legal draft.
* **Advocatus Diaboli:** Actively searches for flaws and counter-arguments in the draft.
* **Advocatus Facti:** Cross-references the response against the provided facts for completeness.
* **Senior Law Consultant:** Synthesizes all perspectives into a final, professional German response.

---

### 📂 File Traversal & Modular Order
To modularize the code, follow this pipeline flow:
1.  **`base.py`**: The foundation (Semantic Chunking logic).
2.  **`KG_Creation.py`**: The construction (Extracting entities and building the Knowledge Graph).
3.  **`system_prompts/`**: The brain (Specific personas like `fact_extract` or `urteile` for specialized tasks).
4.  **`Graph_RAG_advanced_RAG.py`**: The engine (Advanced retrieval using TF-IDF and contextual neighbor nodes).

---

### ⚠️ Troubleshooting & Version Compatibility
* **LlamaIndex v0.10+ Migration:** All core imports now use the `llama_index.core` namespace. Integrations for Groq and Hugging Face must be installed as separate packages.
* **Dependency Resolution:** If `ImportError: cannot import name 'PreTrainedModel' from 'transformers'` occurs, it indicates a version mismatch.
* **Fix Command:**
    ```bash
    pip install --upgrade transformers sentence-transformers llama-index-core llama-index-embeddings-huggingface llama-index-llms-groq
    ```

In [3]:
def combine_sentences(sentences: List[str], buffer_size: int = 1) -> List[str]:
    """Combine sentences.

    Ported over from:
    https://github.com/FullStackRetrieval-com/RetrievalTutorials/blob/main/5_Levels_Of_Text_Splitting.ipynb

    """
    # Go through each sentence dict
    for i in range(len(sentences)):
        # Create a string that will hold the sentences which are joined
        combined_sentence = ""

        # Add sentences before the current one, based on the buffer size.
        for j in range(i - buffer_size, i):
            # Check if the index j is not negative (to avoid index out of range like on the first one)
            if j >= 0:
                # Add the sentence at index j to the combined_sentence string
                combined_sentence += sentences[j]["sentence"] + " "

        # Add the current sentence
        combined_sentence += sentences[i]["sentence"]

        # Add sentences after the current one, based on the buffer size
        for j in range(i + 1, i + 1 + buffer_size):
            # Check if the index j is within the range of the sentences list
            if j < len(sentences):
                # Add the sentence at index j to the combined_sentence string
                combined_sentence += " " + sentences[j]["sentence"]

        # Then add the whole thing to your dict
        # Store the combined sentence in the current sentence dict
        sentences[i]["combined_sentence"] = combined_sentence

    return sentences


### 🧩 Function Breakdown: `combine_sentences`

The `combine_sentences` function implements a **Sliding Window Context Buffer**. It is designed to enhance the "meaning" of individual sentences before they are turned into vector embeddings.

#### 1. The Input
It takes a list of dictionaries called `sentences`. Each dictionary contains the raw text of a single sentence.

#### 2. The Buffer Logic (`buffer_size`)
The `buffer_size` determines how many sentences from the "past" and "future" are merged with the "current" sentence.
* If `buffer_size = 1`: The function joins the **previous** sentence, the **current** sentence, and the **next** sentence.
* If `buffer_size = 2`: It joins the **two previous**, the **current**, and the **two next** sentences.

#### 3. Step-by-Step Execution
1. **Looping:** It iterates through every sentence in your document one by one.
2. **Pre-pending:** It looks at the index `i - buffer_size`. If it exists (is not negative), it adds that text to a temporary string.
3. **Current:** It adds the actual sentence at index `i`.
4. **Post-pending:** It looks at the index `i + buffer_size`. If it exists (is not beyond the end of the file), it adds that text.
5. **Storage:** It saves this new, longer string into a new key in the dictionary called `combined_sentence`.

#### 4. Why is this necessary?
* **Semantic Accuracy:** Embedding models work better on paragraphs than on isolated sentences.
* **Topic Transition:** By having "overlap" between sentences, the system can more accurately calculate the **Cosine Distance** (semantic shift) between one point in the text and the next. This allows the `SemanticChunker` to find the perfect place to cut the text into a chunk."

In [4]:
def calculate_cosine_distances(sentences: List[str]) -> List[float]:
    """Calculate cosine distances."""
    from sklearn.metrics.pairwise import cosine_similarity

    distances: List[float] = []
    for i in range(len(sentences) - 1):
        embedding_current = sentences[i]["embedding"]
        embedding_next = sentences[i + 1]["embedding"]

        # Calculate cosine similarity
        similarity = cosine_similarity([embedding_current], [embedding_next])[0][0]

        # Convert to cosine distance
        distance = 1 - similarity

        # Append cosine distance to the list
        distances.append(distance)

    # add last distance (just put 0)
    distances.append(0)

    return distances

### 📉 Function Breakdown: `calculate_cosine_distances`

The `calculate_cosine_distances` function performs the mathematical comparison needed to detect topic shifts. It transforms abstract "meaning" into a numerical value that the computer can use to make splitting decisions.

#### 1. How it works (Step-by-Step)
1. **Pairwise Comparison:** It loops through the list of sentences, looking at a "Current" sentence ($i$) and the "Next" sentence ($i + 1$).
2. **Retrieve Embeddings:** It pulls the high-dimensional vector (the "embedding") created for each sentence.
3. **Cosine Similarity:** It uses `sklearn` to calculate the **Cosine Similarity**. 
    * A score of **1.0** means the sentences are identical in meaning.
    * A score of **0.0** means they are completely unrelated.
4. **Distance Conversion:** It converts similarity into **Distance** using the formula:  
   $$\text{Distance} = 1 - \text{Similarity}$$
    * **Low Distance (near 0):** The sentences are semantically very close (same topic).
    * **High Distance (near 1):** The sentences are semantically far apart (topic shift).
5. **Final Padding:** It adds a `0` at the very end of the list because the last sentence has no "next" sentence to compare against.

#### 2. Why use Cosine Distance?
In NLP (Natural Language Processing), we don't care how many words are shared; we care about the **angle** of the meaning. Cosine distance is the industry standard for measuring this because:
* It is independent of sentence length.
* It captures the "contextual relationship" between thoughts rather than just keyword matching.

#### 3. Role in the Pipeline
This function produces a "Distance Map" of the document. Later, the `get_indices_above_threshold` function will look at these distances to find the "peaks"—the points where the distance is so high that the AI decides to cut the text and start a new chunk.

In [5]:
def get_indices_above_threshold(distances: List[float], threshold: float) -> List[int]:
    """Get indices above threshold."""
    # We need to get the distance threshold that we'll consider an outlier
    # We'll use numpy .percentile() for this
    breakpoint_distance_threshold = np.percentile(
        distances, threshold
    )  # If you want more chunks, lower the percentile cutoff

    # Then we'll get the index of the distances that are above the threshold. This will tell us where we should split our text
    return [
        i for i, x in enumerate(distances) if x > breakpoint_distance_threshold
    ]  # The indices of those breakpoints on your list

### ✂️ Function Breakdown: `get_indices_above_threshold`

The `get_indices_above_threshold` function identifies the **Breakpoints** where the text will be split. It uses a statistical approach to distinguish between a "normal" flow of sentences and a "significant" change in topic.

#### 1. How it works (Step-by-Step)
1.  **Input:** It takes the list of `distances` (from the cosine calculation) and a `threshold` (usually a percentile like 95.0).
2.  **Percentile Calculation:** It uses `np.percentile(distances, threshold)` to find a "cutoff" value. 
    * If the threshold is **95**, it finds the distance value that is higher than 95% of all other distances in the document.
3.  **Filtering:** It iterates through the list of distances and checks: *"Is this specific distance higher than our 95th percentile cutoff?"*
4.  **Output:** It returns a list of **Indices** (positions) where these high-distance "peaks" occurred.

#### 2. Why use Percentiles instead of a fixed number?
* **Adaptability:** Different documents have different "natural" flow. A legal contract might have very high distances between clauses, while an essay might have very low distances.
* **Dynamic Splitting:** By using percentiles, the function adapts to the document's specific "meaning density." It always picks the most significant shifts relative to the rest of that specific text.

#### 3. Adjusting the Chunks
* **Lower Percentile (e.g., 80):** Makes the "bar" lower. More distances will be considered "high," resulting in **more, smaller chunks**.
* **Higher Percentile (e.g., 98):** Makes the "bar" higher. Only the most extreme topic shifts will trigger a split, resulting in **fewer, larger chunks**.

#### 4. Role in the Pipeline
This function provides the "map of cuts." The next function in the chain (`make_chunks`) will use these indices to actually slice the string into the final pieces used by the RAG system.

In [6]:
def make_chunks(sentences: List[str], indices_above_thresh: List[int]) -> List[str]:
    """Make chunks."""
    start_index = 0
    chunks = []
    # Iterate through the breakpoints to slice the sentences
    for index in indices_above_thresh:
        # The end index is the current breakpoint
        end_index = index

        # Slice the sentence_dicts from the current start index to the end index
        group = sentences[start_index : end_index + 1]
        combined_text = " ".join([d["sentence"] for d in group])
        chunks.append(combined_text)

        # Update the start index for the next group
        start_index = index + 1

    # The last group, if any sentences remain
    if start_index < len(sentences):
        combined_text = " ".join([d["sentence"] for d in sentences[start_index:]])
        chunks.append(combined_text)

    return chunks

### ✂️ Function Breakdown: `make_chunks`

The `make_chunks` function is the final step in the **Semantic Chunking** pipeline. It performs the physical "slicing" of the document based on the semantic breakpoints identified in the previous steps.

#### 1. How it works (Step-by-Step)
1. **Inputs:** It receives the full list of `sentences` and the `indices_above_thresh` (the list of "peaks" where the topic changed).
2. **The Loop:** It iterates through every breakpoint index:
   * **Slicing:** It takes all sentences from the `start_index` up to the current `breakpoint`.
   * **Joining:** It joins those individual sentences back together into a single string using `" ".join()`.
   * **Appending:** It adds that completed string (a "chunk") to the `chunks` list.
3. **Updating Pointer:** It moves the `start_index` to the position immediately after the cut.
4. **The "Tail" Check:** After the loop, it checks if there are any sentences left over (from the last breakpoint to the end of the file). If so, it groups them into one final chunk.

#### 2. Why is this logic specific?
* **Sentence Preservation:** It ensures that we never cut a sentence in half. A chunk always ends at the end of a full sentence.
* **Semantic Integrity:** Because the `indices_above_thresh` were chosen based on topic shifts, each string in the resulting list represents a single, coherent legal concept or fact.

#### 3. Role in the RAG Pipeline
These chunks are what LlamaIndex calls **Nodes**. Each string returned by this function becomes a searchable unit in your Vector Store. When a user asks a question, the system retrieves these specific chunks because they contain contextually related information.

In [7]:
class SemanticChunker(MetadataAwareTextSplitter):
    """Semantic splitter using HuggingFace instead of OpenAI."""

    buffer_size: int = Field(
        default=1, description="Number of sentences to include in each chunk."
    )
    # Use BaseEmbedding here so any LlamaIndex embedding model works
    embed_model: Optional[Any] = Field(
        default=None, description="Embedding model."
    )
    breakpoint_percentile_threshold: float = Field(
        default=95.0,
        description="Percentile threshold for breakpoint distance.",
    )

    def __init__(
        self,
        buffer_size: int = 1,
        embed_model: Optional[Any] = None,
        breakpoint_percentile_threshold: float = 95.0,
        **kwargs: Any
    ):
        # Local import to avoid startup overhead
        from llama_index.embeddings.huggingface import HuggingFaceEmbedding

        super().__init__(
            buffer_size=buffer_size,
            # If no model is provided, it defaults to HuggingFace
            embed_model=embed_model or HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
            breakpoint_percentile_threshold=breakpoint_percentile_threshold,
            **kwargs
        )

    @classmethod
    def class_name(cls) -> str:
        return "SemanticChunker"

    def split_text_metadata_aware(self, text: str, metadata_str: str) -> List[str]:
        return self._split_text(text)

    def split_text(self, text: str) -> List[str]:
        return self._split_text(text)

    def _split_text(self, text: str) -> List[str]:
        single_sentences_list = re.split(r"(?<=[.?!])\s+", text)
        sentences = [
            {"sentence": x, "index": i} for i, x in enumerate(single_sentences_list)
        ]

        combined_sentences = combine_sentences(sentences, self.buffer_size)

        # Logic remains the same, but now uses HF model
        embeddings = self.embed_model.get_text_embedding_batch(
            [x["combined_sentence"] for x in combined_sentences]
        )
        
        for i, embedding in enumerate(embeddings):
            combined_sentences[i]["embedding"] = embedding

        distances = calculate_cosine_distances(combined_sentences)
        indices_above_thresh = get_indices_above_threshold(
            distances, self.breakpoint_percentile_threshold
        )

        return make_chunks(combined_sentences, indices_above_thresh)

### 🧠 Class Breakdown: `SemanticChunker`

The `SemanticChunker` class is the high-level controller that manages the entire semantic splitting workflow. It converts raw text into meaningful, context-aware nodes.

#### 1. Key Configuration (The Parameters)
* **`buffer_size`**: The "context window" size. It determines how many sentences are grouped together before calculating meaning.
* **`embed_model`**: The AI model (OpenAI or HuggingFace) used to turn text into mathematical vectors.
* **`breakpoint_percentile_threshold`**: The sensitivity of the splitter. A higher value (e.g., 95) results in larger chunks; a lower value results in more frequent splits.

#### 2. The Internal Engine: `_split_text`
This is the heart of the class. When you give it a document, it executes these 6 steps in order:
1.  **Regex Splitting:** Uses regular expressions to cut the text into a clean list of individual sentences.
2.  **Context Enrichment:** Calls `combine_sentences` to add "buffer" context to every sentence.
3.  **Vectorization (Embedding):** Sends the buffered sentences to the Embedding Model to get their numerical "meaning" vectors.
4.  **Distance Mapping:** Calls `calculate_cosine_distances` to see how much the topic changes between every pair of sentences.
5.  **Peak Detection:** Calls `get_indices_above_threshold` to find the exact spots where the topic changed significantly.
6.  **Assembly:** Calls `make_chunks` to cut the text at those peaks and return the final list of strings.

#### 3. Why is this better than basic splitting?
Standard splitters (like RecursiveCharacterSplitter) cut text based on **size** (e.g., "split every 500 characters"). This often cuts a legal argument right in the middle, making the RAG system lose context. 
The `SemanticChunker` cuts text based on **intent**. It only splits when the "math" shows that the author has started talking about something new.

#### 4. Role in the Pipeline
This class is a **Node Parser**. In a LlamaIndex workflow, you pass this object to a `ServiceContext` or `Settings`. It ensures that every "Node" in your Vector Database is a semantically complete idea, which significantly improves the accuracy of the answers your AI provides.

### 🔄 Evolutionary Change: Switching to HuggingFace Native Chunker

In this version of the `SemanticChunker`, we have successfully decoupled the code from OpenAI's proprietary ecosystem. 

#### What changed?
1. **Default Model:** The `__init__` method now defaults to `HuggingFaceEmbedding` rather than `OpenAIEmbedding`.
2. **Local Processing:** Embeddings are now generated on your local CPU/GPU. This is significantly better for **Legal Data Privacy**, as the text content never leaves your local environment to be vectorized.
3. **Type Flexibility:** By using `Any` or `BaseEmbedding` in the Field definitions, we ensure the class is compatible with any open-source model you choose to download from the Hugging Face Hub.

#### Why is this better?
* **Cost:** Zero API costs for chunking.
* **Privacy:** Essential for legal documents where data sovereignty is a priority.
* **Offline Capability:** You can run this chunking logic without an internet connection once the model is downloaded.

In [8]:
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core.llama_pack import BaseLlamaPack
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import os
from typing import List, Dict, Any

class SemanticChunkingQueryEnginePack(BaseLlamaPack):
    """Semantic Chunking Query Engine Pack.

    Takes in a list of documents, parses it with semantic embedding chunker,
    and runs a query engine on the resulting chunks.
    """

    def __init__(
        self,
        documents: List[Any],
        buffer_size: int = 1,
        breakpoint_percentile_threshold: float = 80.0,
        persist_dir: str = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vectorstores\text" 
    ) -> None:
        """Init params."""
        self.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
        
        # 🔥 THE FIX: Check for the specific core file (docstore.json) 
        # to verify the Vector DB is actually healthy and usable.
        index_exists = os.path.exists(os.path.join(persist_dir, "docstore.json"))

        if index_exists:
            # 1. Load the existing index from the storage directory
            print(f"✅ Status: Found existing index files. Loading Vector DB from {persist_dir}...")
            storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
            self.vector_index = load_index_from_storage(storage_context)
            
            # Re-initialize the splitter so it's available in get_modules()
            self.splitter = SemanticChunker(
                buffer_size=buffer_size,
                breakpoint_percentile_threshold=breakpoint_percentile_threshold,
                embed_model=self.embed_model,
            )
        else:
            # 2. If the index files don't exist, we must create them
            if not documents:
                print("❌ Error: No documents provided. Database cannot be created.")
                return

            print("⏳ Status: No existing index found. Starting semantic chunking and embedding...")
            self.splitter = SemanticChunker(
                buffer_size=buffer_size,
                breakpoint_percentile_threshold=breakpoint_percentile_threshold,
                embed_model=self.embed_model,
            )

            # Process the text into semantic nodes
            nodes = self.splitter.get_nodes_from_documents(documents)
            
            # Create the Vector Index
            self.vector_index = VectorStoreIndex(nodes)
            
            # 🔥 THE FIX: Explicitly create the directory structure before persisting
            # This ensures that even if 'vectorstores' doesn't exist, it gets created.
            os.makedirs(persist_dir, exist_ok=True)
            
            # Persist the index to disk for future use
            self.vector_index.storage_context.persist(persist_dir=persist_dir)
            print(f"🚀 Status: Successfully created and saved new Vector DB to {persist_dir}")
            
        # Finalize the query engine
        self.query_engine = self.vector_index.as_query_engine()

    def get_modules(self) -> Dict[str, Any]:
        return {
            "vector_index": self.vector_index,
            "query_engine": self.query_engine,
            "splitter": self.splitter,
            "embed_model": self.embed_model,
        }

    def run(self, query: str) -> Any:
        """Run the pipeline."""
        return self.query_engine.query(query)

### 🧠 Custom LlamaPack: Semantic Chunking & Persistence Engine

This class, `SemanticChunkingQueryEnginePack`, handles the end-to-end orchestration of our document processing pipeline.

#### 🛠️ Key Features:
* **BAAI/bge-small-en-v1.5 Embeddings**: Utilizes a high-performance, local embedding model for semantic representation.
* **Intelligent Persistence**: Automatically detects if a Vector DB already exists on disk (via `docstore.json`). If found, it loads the index in seconds; otherwise, it triggers a new embedding run.
* **Semantic Chunker**: Replaces traditional fixed-size chunking with "meaning-based" splitting, ensuring legal concepts remain contextually intact.
* **Automatic Directory Management**: Uses `os.makedirs` to ensure the complex local path for the vector store is created before saving.

#### 📂 Storage Structure:
The resulting database is stored at: 
`C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vectorstores\text`

### Building the Graphs

### 🕸️ Module Breakdown: `Knowledge Graph Creation`

This script serves as the "Factory" that builds the Knowledge Graph. It moves from raw text to a structured, interactive network.

#### 1. Data Processing
* **Semantic Ingestion:** Uses `SemanticChunker` to ensure legal concepts aren't split mid-sentence.
* **LLM Extraction:** Uses GPT-4o-mini to identify 'Node 1', 'Node 2', and the 'Edge' (relationship) between them.

#### 2. Relationship Engineering
* **Weighting:** Concepts mentioned together multiple times receive higher weights.
* **Contextual Proximity:** Automates connections between terms appearing in the same paragraph to capture implicit relationships.

#### 3. Graph Theory & Analytics
* **NetworkX:** The engine used to store nodes and edges.
* **Girvan-Newman Algorithm:** Automatically detects "Legal Modules" or clusters within the text without manual tagging.
* **Persistence:** Saves the graph as `.json`, `.csv`, and `.pkl` (pickle) for use in the RAG retrieval scripts.

#### 4. Interactive Visualization
* **Pyvis:** Generates a 3D-like physics-based visualization in `docs/index.html`. 
* **Node Color:** Represents the detected community.
* **Node Size:** Represents the "Importance" (Degree Centrality) of the term in the corpus.

### 💾 Persistent Vector Storage

We have modified the `SemanticChunkingQueryEnginePack` to support **on-disk persistence**. 

#### 🔧 Key Updates:
* **StorageContext:** We now utilize LlamaIndex's storage context to map the in-memory index to a physical directory.
* **Conditional Loading:** The engine now checks for the existence of `./vectorstores/text`. If found, it loads the data instantly, bypassing the chunking and embedding phase.
* **Alignment:** This setup ensures that the text corpus used by our **Senior Law Consultant** is identical to the one referenced in our **Knowledge Graph** metadata.

In [9]:
import glob
import json
import requests

from langchain_community.document_loaders import PyPDFLoader, UnstructuredPDFLoader, PyPDFium2Loader
from langchain_community.document_loaders import PyPDFDirectoryLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import random
import networkx as nx
import seaborn as sns
from pyvis.network import Network
import json
from yachalk import chalk
import uuid

import io
import pickle

### 📚 Library Overview: Graph RAG Setup

This block initializes the environment for a **Graph-Augmented Retrieval** system. The imports are categorized as follows:

1.  **Ingestion & Splitting:** `langchain` loaders handle PDF/Text input, while `RecursiveCharacterTextSplitter` breaks text into chunks for the LLM to analyze.
2.  **The Knowledge Graph:** `networkx` is used as the mathematical backend to store legal entities and their relationships.
3.  **Visual Intelligence:** `pyvis` and `seaborn` are used to create interactive, color-coded maps of the legal network.
4.  **Utilities:** `requests` handles the API bridge to the LLM (OpenAI/Groq), and `pickle` handles the saving/loading of the generated graph data.
5.  **Data Integrity:** `uuid` ensures every chunk of the Indian Constitution or Case Law has a unique fingerprint.

In [10]:
from llama_index.core.llama_pack import download_llama_pack
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

api_key = os.getenv("GROQ_API_KEY")

In [11]:
from llama_index.core.llms import ChatMessage
from llama_index.core import Settings

def generate_groq(prompt):
    try:
        # Create the structured messages for LlamaIndex
        messages = [
            ChatMessage(
                role="system", 
                content=(
                    "You are a network graph maker who extracts terms and their relations from a given context. "
                    "The purpose of the Knowledge graph will be to structure legal areas for later query purposes. "
                    "Your task is to extract the ontology of terms mentioned in the context. \n"
                    "Thought 1: Identify key atomistic terms (person, organization, concept, document, etc.).\n"
                    "Thought 2: Identify one-on-one relations between terms mentioned in the same context.\n"
                    "Thought 3: Find the specific relation for each pair.\n"
                    "Format: A list of JSON objects with 'node_1', 'node_2', and 'edge'."
                )
            ),
            ChatMessage(role="user", content=prompt)
        ]

        # Use the global LLM from Settings
        # json_mode=True ensures Llama 3.3 outputs valid JSON
        response = Settings.llm.chat(messages, response_format={"type": "json_object"})

        # LlamaIndex returns a ChatResponse object; we want the message content
        return response.message
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

### 🤖 Function Breakdown: `generate_groq` (The Relationship Extractor)

This function is the bridge between **Unstructured Text** and a **Structured Graph**. 

#### 🧩 How it works:
* **The "System" Role:** We instruct the LLM (Llama 3.3 via Groq) to act specifically as an ontology extractor. It ignores the "story" and focuses on the "structure."
* **Triplets:** The output is always in the form of a **Triplet**: `(Node 1) --[Edge]--> (Node 2)`. 
  * *Example:* `(John Smith) --[filed]--> (Lawsuit)`.
* **JSON Enforcement:** We utilize Groq's JSON mode to ensure the output can be directly converted into a Python list without manual cleaning.

#### ⚖️ Legal Relevance:
For the **Indian Constitution** or **Smith v. Johnson**, this function ensures that legal principles are linked correctly. For example, it will link "Negligence" to "Defendant" and "Contract" to "Article 300A" based on the context provided.

### 🔄 Upgrading the LLM Provider: LlamaIndex vs. LangChain

Instead of using the raw `groq` Python client, we have shifted to a high-level framework integration. 

#### Why we prefer `llama_index.llms.groq`:
1. **Global Consistency:** By setting `Settings.llm = Groq(...)`, every component in our notebook (the Splitter, the Index, and the Graph Extractor) uses the same fast Groq backbone.
2. **Unified Interface:** We use `.complete()` or `.chat()` methods which handle retries and error logging automatically.
3. **Optimized for RAG:** LlamaIndex is purpose-built for the "Retrieval" part of our Legal Graph system, making it easier to link Article 300A to our case chunks.

### 🛠️ Function Upgrade: Native LlamaIndex + Groq Integration

By utilizing the `llama_index.llms.groq` library, we have simplified our extraction function:

1. **Global Settings:** The function now draws from `Settings.llm`, meaning we only have to manage our API key and model name in one place.
2. **Type Safety:** We use the `ChatMessage` class, which is the standard communication format for all LlamaIndex components.
3. **JSON Mode Reliability:** We pass the `response_format` directly into the `.chat()` call, leveraging Llama 3.3's specialized ability to output structured JSON data for our graph nodes.

In [12]:
import json

def graphPrompt(input_text: str, metadata={}, model="llama-3.3-70b-versatile"):
    # Create the prompt for the LLM
    USER_PROMPT = f"context: {input_text} \n\n output: "
    
    # Call your previously defined generate_groq function
    response = generate_groq(USER_PROMPT)

    try:
        # LlamaIndex returns a ChatResponse; we need the content attribute
        if response and hasattr(response, 'content'):
            content = response.content

            # Clean up Markdown formatting if the model adds it
            content_cleaned = content.replace('```json', '').replace('```', '').strip()

            # Parse string into Python list
            result = json.loads(content_cleaned)

            # Inject metadata (like chunk_id) into every relationship
            # If result is a single dict, wrap it in a list; otherwise process the list
            if isinstance(result, dict):
                result = [result]
                
            result = [dict(item, **metadata) for item in result]
            return result
        else:
            return None # Return None so .dropna() can handle it later
            
    except json.JSONDecodeError as e:
        # Only print errors if you are debugging; otherwise it clutters the console
        # print(f"JSON Parsing Error: {e}") 
        return None
    except Exception as e:
        print(f"Unexpected Error in graphPrompt: {e}")
        return None

### 🧹 Function Breakdown: `graphPrompt` (The Sanitizer)

The `graphPrompt` function ensures that the AI's "Chat" output is converted into "Machine-Readable" data.

#### 1. Input Handling
It accepts the raw text chunk and a `metadata` dictionary. In a Legal RAG, the metadata usually contains the `case_number` or `article_index`.

#### 2. Cleaning Logic
LLMs are "chatty" by nature. Even in JSON mode, they might include:
* Markdown headers (e.g., ` ```json `)
* Trailing whitespace
* Explanation text
This function strips all "noise" until only the valid JSON array remains.

#### 3. Metadata Injection
This is the most important part for **Legal Accountability**. 
* **Without it:** You have a graph that says "John Smith sued Michael Johnson."
* **With it:** The graph says "John Smith sued Michael Johnson **[Source: Paragraph 4 of Smith v. Johnson]**."
This ensures every node in your Knowledge Graph has a verifiable "paper trail."

**Purpose:** Processes a single text chunk to extract a list of "Triplets" (Node 1, Node 2, Edge).

**Key Features:**
* **Context Wrapping:** Packages the raw text with a specialized "Network Graph Maker" system prompt.
* **JSON Sanitization:** Uses string manipulation to strip away Markdown artifacts (like ` ```json `) that LLMs often include.
* **Metadata Binding:** Injects the unique `chunk_id` into every extracted relationship, ensuring total traceability back to the source document.
* **Error Handling:** Gracefully returns `None` if the JSON is malformed, preventing the entire pipeline from crashing.

In [13]:
# function for converting a collection of document chunks into a structured pandas DataFrame
import pandas as pd
def documents2Dataframe(documents) -> pd.DataFrame:
    rows = []
    for chunk in documents:
        row = {
            "text": chunk.text,
            **chunk.metadata,
            "chunk_id": uuid.uuid4().hex,
        }
        rows = rows + [row]

    df = pd.DataFrame(rows)
    return df

### 📊 Function Breakdown: `documents2Dataframe` (The Tabulator)

This function transforms a list of AI `Document` objects into a structured **Pandas DataFrame**. This is a prerequisite for the Knowledge Graph extraction phase.

#### 🛠 Key Components:
* **Text Extraction:** Captures the actual legal text from each chunk.
* **Metadata Preservation:** Automatically pulls in source info (e.g., filename, author) using Python's dictionary unpacking (`**`).
* **Unique ID Generation (`chunk_id`):** Assigns a unique UUID to every chunk. This is the **Primary Key** that connects our final Knowledge Graph back to the original source text.

#### ⚖️ Legal RAG Impact:
By converting chunks to a DataFrame, we can:
1. **Filter Content:** Easily remove empty or irrelevant chunks before sending them to Groq.
2. **Batch Process:** Use the `.apply()` method to run the Knowledge Graph extraction across the entire case file in one line of code.
3. **Exportability:** Save the intermediate "chunks" as a CSV for manual legal audit.

In [14]:
import numpy as np
import pandas as pd

def df2Graph(dataframe: pd.DataFrame, model="llama-3.3-70b-versatile") -> list:
    # 1. Apply the extraction logic to every row (chunk) in the DataFrame
    # This calls graphPrompt for each chunk of text
    results = dataframe.apply(
        lambda row: graphPrompt(row.text, {"chunk_id": row.chunk_id}, model), axis=1
    )
    
    # 2. Remove any rows where the LLM failed to return a valid graph (None values)
    results = results.dropna()
    
    # 3. SAFETY CHECK: Ensure we actually have data before concatenating
    if results.empty or len(results) == 0:
        print("⚠️ WARNING: No relationships were extracted from the text.")
        print("Check if your data_input files have content or if your API key is valid.")
        return []

    # 4. Flatten the list of lists into a single master list of triplets
    try:
        results_list = results.tolist()
        concept_list = np.concatenate(results_list).ravel().tolist()
        
        print(f" Success: Total relationships extracted via Groq: {len(concept_list)}")
        return concept_list
    except ValueError as e:
        print(f" Error during concatenation: {e}")
        return []

### ⛓️ Function Breakdown: `df2Graph` (The Logic Orchestrator)

This function acts as the "Batch Processor" for the Knowledge Graph. It converts a table of text chunks into a master list of legal relationships.

#### 🛠 Key Steps:
* **Batch Execution:** Instead of manual loops, it uses `dataframe.apply()` to send chunks to the LLM efficiently.
* **Failure Handling:** Automatically filters out chunks where the LLM couldn't find a clear relationship (the `.dropna()` step).
* **Dimensionality Reduction:** It "flattens" the nested lists of nodes into a single master list of legal triplets.
* **Provenance Tracking:** By passing the `chunk_id` into the `graphPrompt`, it maintains the link between the **Knowledge Graph** and the **Source Text**.

#### ⚖️ Relevance to Legal RAG:
This function is where the "heavy lifting" happens. For a document like the **Indian Constitution**, this function is responsible for making sure every single Article is visited and turned into a set of related nodes (e.g., `Article 21` ➡️ `Right to Life`).

**Purpose:** Orchestrates the `graphPrompt` function across all rows of the document DataFrame.

**Key Features:**
* **Efficient Processing:** Uses the Pandas `.apply()` method to process the entire case file (or Constitution) in a single pass.
* **Result Filtering:** Uses `.dropna()` to remove any failed extractions, ensuring only valid legal data moves forward.
* **Concatenation Safety:** Includes a defensive check to verify data exists before attempting `np.concatenate`, resolving the common `ValueError` associated with empty datasets.
* **Audit Logging:** Outputs the final count of successful extractions to the console for real-time monitoring of the ingestion process.

In [15]:
def graph2Df(nodes_list) -> pd.DataFrame:
    # 1. Create DataFrame from the list of dictionaries
    graph_dataframe = pd.DataFrame(nodes_list)
    
    # 2. Safety Check: If the list was empty, return an empty DataFrame with correct columns
    if graph_dataframe.empty:
        print("⚠️ Warning: nodes_list was empty. Creating empty DataFrame.")
        return pd.DataFrame(columns=["node_1", "node_2", "edge", "chunk_id"])

    # 3. Handle potential whitespace and replace with NaN
    graph_dataframe = graph_dataframe.replace(" ", np.nan)
    
    # 4. Safety Check: Ensure the required columns actually exist
    required_cols = ["node_1", "node_2"]
    existing_cols = [col for col in required_cols if col in graph_dataframe.columns]
    
    if len(existing_cols) < 2:
        print(f"❌ Error: LLM output missing required keys. Found columns: {graph_dataframe.columns.tolist()}")
        # Return an empty DF to prevent crashing the rest of the script
        return pd.DataFrame(columns=["node_1", "node_2", "edge", "chunk_id"])

    # 5. Clean and Normalize
    graph_dataframe = graph_dataframe.dropna(subset=["node_1", "node_2"])
    graph_dataframe["node_1"] = graph_dataframe["node_1"].astype(str).apply(lambda x: x.lower())
    graph_dataframe["node_2"] = graph_dataframe["node_2"].astype(str).apply(lambda x: x.lower())

    return graph_dataframe

### 🧹 Function Breakdown: `graph2Df` (The Graph Normalizer)

This function performs the final "Post-Processing" on our extracted legal relationships. It ensures the data is clean enough for mathematical network analysis.

#### 🛠 Key Cleaning Steps:
* **Empty Value Removal:** Identifies and deletes "broken" relationships where the LLM failed to provide a starting or ending node.
* **Normalization:** Converts all entities to **lowercase**. This is essential for "Entity Resolution"—ensuring that 'Breach of Contract' and 'breach of contract' are recognized as the exact same legal concept.
* **Validation:** Prepares the data for `NetworkX` by ensuring every row is a valid pair of connected concepts.

#### ⚖️ Legal RAG Impact:
In the **Smith v. Johnson** case, this function ensures that every time "John Smith" is mentioned (regardless of capitalization), all those facts are attached to a **single node**. This allows the RAG system to "see" the full picture of the Plaintiff's actions across the entire document.

In [16]:
# function to calculate contextual_proximity based on chunk alone
def contextual_proximity(df: pd.DataFrame) -> pd.DataFrame:
    # Melt the dataframe into a list of nodes
    dfg_long = pd.melt(
        df, id_vars=["chunk_id"], value_vars=["node_1", "node_2"], value_name="node"
    )
    dfg_long.drop(columns=["variable"], inplace=True)

    # Self join with chunk id as the key will create a link between terms occuring in the same text chunk
    dfg_wide = pd.merge(dfg_long, dfg_long, on="chunk_id", suffixes=("_1", "_2"))
    # drop self loops
    self_loops_drop = dfg_wide[dfg_wide["node_1"] == dfg_wide["node_2"]].index
    dfg2 = dfg_wide.drop(index=self_loops_drop).reset_index(drop=True)

    # Ensure node_1 is always less than or equal to node_2 to treat pairs back and forth the same
    dfg2["node_1"], dfg2["node_2"] = np.where(
        dfg2["node_1"] <= dfg2["node_2"],
        [dfg2["node_1"], dfg2["node_2"]],
        [dfg2["node_2"], dfg2["node_1"]]
    )

### 🧠 Function Breakdown: `contextual_proximity` (The Logic Inferrer)

This function discovers "hidden" relationships by analyzing which terms co-occur within the same semantic chunk.

#### 🛠 How it Works:
1. **Context Mapping:** It groups all entities extracted from a single paragraph or section.
2. **Combinatorial Pairing:** It creates a link between every possible pair of terms within that section.
3. **De-duplication:** It ensures that the direction of the link (A to B vs B to A) doesn't result in duplicate data.

#### ⚖️ Legal RAG Impact:
This is essential for **Legal Discovery**. 
* **Scenario:** A case mentions "Limited Liability" and "Software Licensing" in the same background section. 
* **The Result:** Even if the LLM didn't find a direct verb connecting them, this function creates a "Proximity Edge." 
* **The Benefit:** When you later ask the RAG system about licensing, it can "hop" to the liability chunks because it knows they were mentioned together.

In [17]:
# function to calculate contextual_proximity based on chunk alone
def contextual_proximity(df: pd.DataFrame) -> pd.DataFrame:
    # Melt the dataframe into a list of nodes
    dfg_long = pd.melt(
        df, id_vars=["chunk_id"], value_vars=["node_1", "node_2"], value_name="node"
    )
    dfg_long.drop(columns=["variable"], inplace=True)

    # Self join with chunk id as the key will create a link between terms occuring in the same text chunk
    dfg_wide = pd.merge(dfg_long, dfg_long, on="chunk_id", suffixes=("_1", "_2"))
    # drop self loops
    self_loops_drop = dfg_wide[dfg_wide["node_1"] == dfg_wide["node_2"]].index
    dfg2 = dfg_wide.drop(index=self_loops_drop).reset_index(drop=True)

    # Ensure node_1 is always less than or equal to node_2 to treat pairs back and forth the same
    dfg2["node_1"], dfg2["node_2"] = np.where(
        dfg2["node_1"] <= dfg2["node_2"],
        [dfg2["node_1"], dfg2["node_2"]],
        [dfg2["node_2"], dfg2["node_1"]]
    )

    # Group and count edges
    dfg2 = (
        dfg2.groupby(["node_1", "node_2"])
        .agg({"chunk_id": [",".join, "count"]})
        .reset_index()
    )
    dfg2.columns = ["node_1", "node_2", "chunk_id", "count"]
    dfg2.replace("", np.nan, inplace=True)
    dfg2.dropna(subset=["node_1", "node_2"], inplace=True)
    # Drop edges with 1 count
    dfg2 = dfg2[dfg2["count"] != 1]
    dfg2["edge"] = "chunk contextual proximity"
    return dfg2

### 🔗 Function Breakdown: `contextual_proximity` (The Inference Engine)

This function discovers "implicit" relationships by analyzing which legal concepts live in the same text chunks.

#### 🛠 Logic Flow:
* **Co-occurrence Mapping:** It assumes that if two terms appear in the same paragraph, they are contextually related.
* **Combinatorial Linking:** Using a "Self-Join," it creates a network of links between all entities within a single semantic chunk.
* **Weighting:** The `count` column acts as the "Strength" of the relationship. The more paragraphs two terms share, the stronger the bond in our Knowledge Graph.
* **Noise Reduction:** By filtering out single co-occurrences, we ensure the graph only reflects significant, recurring legal themes.

#### 📍 Difference from LLM Extraction:
* **LLM Extraction:** Finds *explicit* verbs (e.g., "X *authorizes* Y").
* **Contextual Proximity:** Finds *thematic* clusters (e.g., "X and Y are always mentioned in the same section of the Constitution").

**Result:** This turns our graph from a simple list of facts into a high-density "Knowledge Web."

In [18]:
# function to calculate global contextual proximity over the whole corpus into a third dfg
def global_contextual_proximity(df: pd.DataFrame) -> pd.DataFrame:

    # Melt the dataframe into a long format
    dfg_long = pd.melt(
        df, id_vars=[], value_vars=["node_1", "node_2"], var_name="variable", value_name="node"
    ).drop(columns=["variable"])

    # Create a self-join using the node column
    dfg_wide = pd.merge(dfg_long.rename(columns={"node": "node_1"}),
                        dfg_long.rename(columns={"node": "node_2"}),
                        left_on="node_1",
                        right_on="node_2")

    # Drop self loops where an edge starts and ends on the same node
    dfg_wide = dfg_wide[dfg_wide["node_1"] != dfg_wide["node_2"]]

    # Group and count the number of co-occurrences of the node pairs
    dfg2 = (
        dfg_wide.groupby(["node_1", "node_2"])
        .size()
        .reset_index(name="count")
    )

    # Filter out pairs with only one co-occurrence
    dfg2 = dfg2[dfg2["count"] > 1]

    # Add edge type
    dfg2["edge"] = "global contextual proximity"

    return dfg2

### 🌍 Function Breakdown: `global_contextual_proximity` (The Global Mapper)

This function identifies high-level legal themes that span across the entire document corpus, moving beyond individual paragraphs.

#### 🛠 Key Logic:
* **Corpus-Wide Analysis:** It ignores chunk boundaries and looks at the frequency of terms across the whole dataset (e.g., all 448+ Articles of the Indian Constitution).
* **Identifying "Hub" Nodes:** It helps detect "Central Pillars" of the law—terms like *Jurisdiction*, *The State*, or *Liability*—that act as connectors between different legal areas.
* **Thematic Clustering:** By finding nodes that co-occur across different files, it allows the RAG system to perform "Transversal Search" (e.g., finding how a property dispute in one case relates to a constitutional right in another).

#### 📍 Practical Example:
If you have 10 different contract cases, the LLM will extract the specific names of the plaintiffs. However, `global_contextual_proximity` will notice that the term **"Compensation"** appears in all 10 cases. It will create a strong "Global Edge" for Compensation, making it a primary entry point for the RAG system when a user asks about money.

In [19]:
# function to apply HI splitter
def custom_hi_text_splitter(directory):
    pages = []

    for file_path in glob.glob(f"{directory}/*.txt"):
        with open(file_path, 'r', encoding='utf-8-sig') as file:
            text = file.read()

            # Find all occurrences of 'HI' followed by numbers
            hi_indices = [match.start() for match in re.finditer(r'HI\d+', text)]
            if hi_indices:
                # Add the first chunk as a Document object
                pages.append(Document(page_content=text[:hi_indices[0]]))

                # Add all subsequent chunks as Document objects
                for start, end in zip(hi_indices, hi_indices[1:] + [None]):
                    pages.append(Document(page_content=text[start:end]))

    return pages

### ✂️ Function Breakdown: `custom_hi_text_splitter` (Domain-Specific Splitting)

This function provides a "Rule-Based" alternative to Semantic Chunking. It is specifically tuned for documents that use "HI" tags as section headers.

#### 🛠 Key Logic:
* **Regex Identification:** Uses `HI\d+` to find structural breakpoints in the raw text.
* **Context Preservation:** By splitting at headers rather than character counts, we ensure that individual legal clauses or case sections remain intact.
* **Compatibility:** Converts the raw text slices into `langchain.core.documents.base.Document` objects so they can be processed by our Knowledge Graph factory.

#### 📍 When to use this vs. Semantic Chunking:
* **Use this:** If your documents are strictly organized (like a codified set of laws or a specific database export).
* **Use Semantic Chunking:** If your documents are "messy" prose (like the *Smith v. Johnson* summary or news articles).

In [20]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

data_dir = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data"
inputdirectory = Path(r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data")

# This is where the output csv files will be written
outputdirectory = Path(r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data_output")
outputdirectory.mkdir(parents=True, exist_ok=True)

if not groq_api_key:
    raise ValueError("❌ GROQ_API_KEY not found. Ensure your .env file is in the same directory as this notebook.")

# 1. Setup your Models
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = Groq(model="llama-3.3-70b-versatile", api_key=groq_api_key)

print("✅ Groq API successfully configured from environment.")

# Initialize an empty string to store the contents of all files
all_content = ""

# Iterate over all .txt files in the directory
for file_path in glob.glob(os.path.join("data_input", "*.txt")):
    with open(file_path, 'r', encoding='utf-8-sig') as file:
        all_content += file.read()

# Using the Settings.embed_model automatically
splitter = SemanticChunker(
    buffer_size=1, 
    breakpoint_percentile_threshold=80, 
    embed_model=Settings.embed_model 
)
# Process Documents

docs = splitter.split_text(all_content)
pages = [Document(page_content=chunk) for chunk in docs]
# ====================

print("Number of chunks = ", len(pages))
#print(pages[0].page_content)


# Convert documents to DataFrame
df = documents2Dataframe(pages)
print(df.shape)
df.head()

# Regenerate the graph with LLM if graph.csv not already made
# Define the path to the graph.csv file
graph_csv_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data_output\graph.csv"

# Check if the graph.csv file exists
if os.path.exists(graph_csv_path):
    regenerate = False
else:
    regenerate = True

if regenerate:
    # Using df2Graph to generate graph data from the DataFrame
    graph_data = df2Graph(df, model='llama-3.3-70b-versatile')  # Using the OpenAI model name
    
    # toDO: Preprocess and Optimize Nodes and Edges
    #optimized_graph_data = optimize_graph_data(graph_data)

    dfg1 = graph2Df(graph_data)  # Converting the graph data to a DataFrame

    # Creating the output directory if it doesn't exist
    if not os.path.exists(outputdirectory):
        os.makedirs(outputdirectory)

    # Writing the graph data and the original DataFrame to CSV files
    dfg1.to_csv(outputdirectory / "graph.csv", sep="|", index=False)
    df.to_csv(outputdirectory / "chunks.csv", sep="|", index=False)
else:
    # Reading the graph data from a CSV file if not regenerating
    dfg1 = pd.read_csv(outputdirectory / "graph.csv", sep="|")


dfg1.replace("", np.nan, inplace=True)
dfg1.dropna(subset=["node_1", "node_2", 'edge'], inplace=True)
dfg1['count'] = 4
# initializing the weight of the direct-relation to 4
# will assign the weight of 1 when later the contextual proximity will be calculated
print(dfg1.shape)
#print(dfg1.head())

dfg2 = contextual_proximity(dfg1)
#print(dfg2.tail())


#toDO: concatenate third Dataframe (global contextual similarity) with only the first dataframe
global_context_df = global_contextual_proximity(dfg1)
# Concatenate the three DataFrames / maybe only conc the dfg1 and the global?
dfg_combined = pd.concat([dfg1, dfg2, global_context_df])
# Group by node pairs. For 'chunk_id' and 'edge', concatenate unique values; sum the 'count'
dfg_final = (
    dfg_combined.groupby(["node_1", "node_2"])
    .agg({
        "chunk_id": lambda x: ','.join(set([str(i) for i in x if pd.notna(i)])),  # Convert to string and concatenate unique, non-null chunk_ids
        "edge": lambda x: ','.join(set([str(i) for i in x if pd.notna(i)])),      # Convert to string and concatenate unique, non-null edge descriptions
        'count': 'sum'                                                            # Sum the weights (counts)
    })
    .reset_index()
)

#toDO: weight adjustement for glaobal proximit to 0.5 - evaluate
# Update the weights for global contextual proximity to 0.5
dfg_final['count'] = np.where(dfg_final['edge'] == 'global contextual proximity', 0.5 * dfg_final['count'], dfg_final['count'])

#concatenate only dfg1 and dfg2 (contextual proximity within chunks)
dfg = pd.concat([dfg1, dfg2], axis=0)
dfg = (
    dfg.groupby(["node_1", "node_2"])
    .agg({"chunk_id": ",".join, "edge": ','.join, 'count': 'sum'})
    .reset_index()
)
dfg
print(dfg.head())

#toDO: either use "dfg" for concatenad dfg1 and dfg2 or "dfg_final" for global_context_df as well
nodes = pd.concat([dfg_final['node_1'], dfg_final['node_2']], axis=0).unique()
nodes.shape

G = nx.Graph()

## Add nodes to the graph
for node in nodes:
    G.add_node(
        str(node)
    )

## Add edges to the graph
for index, row in dfg.iterrows():
    G.add_edge(
        str(row["node_1"]),
        str(row["node_2"]),
        title=row["edge"],
        weight=row['count']/4,
        chunk_id=row['chunk_id'] #toDO check if this actually worked
    )

# Convert the NetworkX graph to a dictionary for JSON
graph_data = nx.node_link_data(G)

# Specify the file path where you want to save the JSON
json_file_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data_output\graph_data.json"

# Write the graph data to a JSON file
with open(json_file_path, "w", encoding="utf-8-sig") as json_file:
    json.dump(graph_data, json_file, ensure_ascii=False)

#toDO: redundant - delete
#Save as pickle file

# Define the file path for saving the pickled graph
pickle_file_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data_output\graph_data.pkl"

# Save the graph to a file using pickle
with open(pickle_file_path, "wb") as pickle_file:
    pickle.dump(G, pickle_file)


# detect communities with GirvanNewman Algo
# 1. Initialize the generator
communities_generator = nx.community.girvan_newman(G)

try:
    # 2. Get the first level (splitting the graph once)
    top_level_communities = next(communities_generator)
    communities = sorted(map(sorted, top_level_communities))
    
    # 3. Try to get a deeper level (splitting further)
    next_level_communities = next(communities_generator)
    communities = sorted(map(sorted, next_level_communities))
    
except StopIteration:
    # If the graph can't be split further, we use the available communities found so far
    print("Notice: Graph is fully partitioned or too small to split further. Using last successful level.")

print("Number of Communities = ", len(communities))

print("Number of Communities = ", len(communities))
#print(communities)

palette = "hls"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Groq API successfully configured from environment.
Number of chunks =  1
(1, 2)
(1, 5)
         node_1        node_2  \
0  organization        person   
1        person  organization   

                                            chunk_id  \
0  603e48b88568492c8abd5ff0e4a19cec,603e48b885684...   
1                   603e48b88568492c8abd5ff0e4a19cec   

                         edge  count  
0  chunk contextual proximity      2  
1                     employs      4  
Notice: Graph is fully partitioned or too small to split further. Using last successful level.
Number of Communities =  2
Number of Communities =  2


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\networkx\readwrite\json_graph\node_link.py:142: FutureWarning: 
The default value will be `edges="edges" in NetworkX 3.6.

To make this warning go away, explicitly set the edges kwarg, e.g.:

  nx.node_link_data(G, edges="links") to preserve current behavior, or
  nx.node_link_data(G, edges="edges") for forward compatibility.
  warnings.warn(


### ✅ Execution Report: Pipeline Validated

The RAG pipeline has successfully completed a full run using the **Groq/HuggingFace** stack.

#### 📈 Results Summary:
* **Embeddings:** `BAAI/bge-small-en-v1.5` successfully generated semantic vectors despite the `position_ids` warning.
* **Extraction:** Groq (Llama 3.3) identified structural relationships (e.g., *Organization -> employs -> Person*).
* **Graph Logic:** Direct and contextual weights were successfully aggregated into a unified edge count.
* **Clustering:** Community detection is initialized, though the current dataset is too small for deep hierarchical partitioning.

#### 🔧 Optimization Note:
The `FutureWarning` in NetworkX has been noted and will be mitigated in the JSON export cell by explicitly defining the `edges` parameter.

In [21]:
def colors2Community(communities) -> pd.DataFrame:
    ## Define a color palette
    p = sns.color_palette(palette, len(communities)).as_hex()
    random.shuffle(p)
    rows = []
    group = 0
    for community in communities:
        color = p.pop()
        group += 1
        for node in community:
            rows += [{"node": node, "color": color, "group": group}]
    df_colors = pd.DataFrame(rows)
    return df_colors

### 🎨 Function Breakdown: `colors2Community` (The Graph Painter)

This function bridges the gap between **Network Analysis** and **Data Visualization**.

#### 🛠 Key Logic:
* **Dynamic Paletting:** Automatically adjusts the number of colors based on how many legal communities were detected by the Girvan-Newman algorithm.
* **Hex Conversion:** Ensures colors are compatible with standard web visualization libraries (HTML/CSS).
* **Node Mapping:** flattens the nested "list of communities" into a searchable DataFrame where every legal term is assigned a visual 'Group' and 'Color'.

#### 📍 Practical Result:
When viewing the final interactive graph, this function ensures that all related nodes (e.g., all entities extracted from *Smith v. Johnson*) share a common color, making it easy to distinguish them from nodes extracted from the *Indian Constitution*.

In [22]:
colors = colors2Community(communities)
colors
print(colors.head())

for index, row in colors.iterrows():
    G.nodes[row['node']]['group'] = row['group']
    G.nodes[row['node']]['color'] = row['color']
    G.nodes[row['node']]['size'] = G.degree[row['node']]

print("Number of nodes in G:", len(G.nodes()))
print("Number of edges in G:", len(G.edges()))

# Print node attributes for a few nodes to verify
for node in list(G.nodes)[:1]:  # Adjust the number of nodes to print as needed
    print(node, G.nodes[node])

# Define the output file path for the HTML content and write to
graph_output_directory = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\docs\index.html"

net = Network(
    notebook=False,
    bgcolor="#1a1a1a",
    cdn_resources="remote",
    height="900px",
    width="100%",
    select_menu=True,
    font_color="#cccccc",
    filter_menu=False,
)

net.from_nx(G)
# net.repulsion(node_distance=150, spring_length=400)
net.force_atlas_2based(central_gravity=0.015, gravity=-31)
# net.barnes_hut(gravity=-18100, central_gravity=5.05, spring_length=380) #alternative physics algo
net.show_buttons(filter_=["physics"])

html = net.generate_html()
with open(graph_output_directory, mode='w', encoding='utf-8-sig') as fp:
        fp.write(html)

#net.show(graph_output_directory, notebook=False)

           node    color  group
0  organization  #db5f57      1
1        person  #57d3db      2
Number of nodes in G: 2
Number of edges in G: 1
organization {'group': 1, 'color': '#db5f57', 'size': 1}


### 🌐 Final Step: Interactive Knowledge Graph Rendering

This block converts our `NetworkX` graph into a high-performance **Pyvis** visualization.

#### 🛠 Key Configuration:
* **Degree-Based Scaling:** Node sizes are dynamically calculated based on their connectivity. Centrally important legal concepts will appear larger than peripheral details.
* **Force-Directed Layout:** We use the `ForceAtlas2` physics engine to ensure that semantically related legal areas (communities) naturally cluster together on the screen.
* **Interactive UI:** The final output includes a search menu and physics toggles, allowing users to explore the relationships between cases and constitutional articles manually.

#### 📂 Export Details:
* **Path:** `./docs/index.html`
* **Format:** Standalone HTML (no Python backend required to view).
* **Styling:** Optimized with a dark-themed UI for professional presentation.

In [23]:
import os
from pathlib import Path

# 1. Test Directory Permissions
test_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vectorstores\text"
try:
    os.makedirs(test_path, exist_ok=True)
    with open(os.path.join(test_path, "test.txt"), "w") as f:
        f.write("test")
    print(f"✅ Directory creation works: {test_path}")
except Exception as e:
    print(f"❌ Directory creation FAILED: {e}")

# 2. Check if documents actually exist
try:
    print(f"📄 Number of documents loaded: {len(documents)}")
except NameError:
    print("❌ Error: The variable 'documents' is not defined at all.")

# 3. Check if the variable 'pack' was initialized
try:
    print(f"🤖 Query Engine Pack initialized: {'vector_index' in pack.get_modules()}")
except NameError:
    print("❌ Error: 'pack' was never created.")

✅ Directory creation works: C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vectorstores\text
❌ Error: The variable 'documents' is not defined at all.
❌ Error: 'pack' was never created.


In [24]:
from llama_index.core import SimpleDirectoryReader

# Ensure this points to the folder containing lawsuit.txt
input_dir = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data" 
documents = SimpleDirectoryReader(input_dir).load_data()
print(f"Loaded {len(documents)} docs") # This MUST be greater than 0

Loaded 404 docs


In [ ]:
from llama_index.core import SimpleDirectoryReader
import os

# 1. Define the input directory
input_dir = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data" 

# 2. Load the documents (This will now work after installing docx2txt)
if os.path.exists(input_dir):
    documents = SimpleDirectoryReader(input_dir).load_data()
    print(f"✅ Loaded {len(documents)} documents from {input_dir}")
else:
    print(f"❌ Error: The directory {input_dir} does not exist.")

# 3. Initialize the Pack (This triggers the folder/file creation)
# Make sure your SemanticChunkingQueryEnginePack class definition cell has been run first!
if len(documents) > 0:
    pack = SemanticChunkingQueryEnginePack(documents=documents)
    print("✅ Pack initialized and VectorStore files should now exist.")
else:
    print("❌ Critical: No documents found. VectorStore cannot be created.")

2026-02-26 21:48:58,799 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


✅ Loaded 404 documents from C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data


2026-02-26 21:48:59,062 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-02-26 21:48:59,087 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-02-26 21:48:59,304 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-02-26 21:48:59,336 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-02-26 21:48:59,562 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-02-26 21:48:59,588 - INFO - HTTP Request: HEAD https://huggingface.c

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-26 21:49:01,455 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-02-26 21:49:01,485 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
2026-02-26 21:49:01,707 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-02-26 21:49:01,731 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.

⏳ Status: No existing index found. Starting semantic chunking and embedding...
🚀 Status: Successfully created and saved new Vector DB to C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vectorstores\text
✅ Pack initialized and VectorStore files should now exist.
